# 2-4. 모델 평가와 threshold 판단 실습

이 notebook은 1장에서 확인한 `feature`와 `label` 전제를 받아 모델 평가 지표로 바꾸는 실습입니다. 수강생은 모델을 튜닝하는 사람이 아니라, 같은 모델과 같은 threshold에서 어떤 데이터가 어떤 변환을 거쳐 어떤 오류 유형을 만들었는지 QA 보고서로 설명해야 하는 담당자입니다.

핵심 흐름은 `raw data → label 표준화 → feature readiness → score → threshold → prediction → label 비교 → metric`입니다. `score`는 Positive class 가능성이고, `threshold`는 score를 class로 바꾸는 기준이며, `prediction`은 최종 예측입니다. 이 경계를 지켜야 `high_risk` 비율 변화가 label 분포인지 prediction 분포인지 혼동하지 않습니다.

| 받은 업무 | 현장 관측값 | 결정 압박 |
| --- | --- | --- |
| 모델 평가 결과를 FP/FN과 threshold 기준으로 설명합니다 | `high_risk` 예측 비율 변화가 보고되었지만 원인이 아직 불명확합니다 | 17시 회의 전까지 모델 자체 문제, 데이터 조건 변화, threshold 검토 중 무엇을 다음 확인으로 남길지 제안해야 합니다 |

| 이번 장에서 만드는 증거 | 보고서에 남길 산출물 | 다음 장으로 넘길 질문 |
| --- | --- | --- |
| data brief, score, prediction, confusion matrix, metric, threshold sweep | `chapter_02_model_metric_packet`, threshold 판단 문장 | 데이터 품질 신호가 metric 변화와 어떻게 연결되는가 |


### 따라하기 안내

이 notebook은 셀을 위에서 아래로 실행하면서 결과를 해석하는 실습입니다. 코드를 모두 이해하는 것보다, 각 셀이 만드는 근거를 보고 QA 판단으로 바꾸는 것이 중요합니다.

**오늘의 목표:** accuracy 하나가 아니라 precision, recall, FP/FN, threshold를 함께 읽습니다.

진행 방법은 단순합니다.

1. 안내 문장을 읽고 이 셀이 무엇을 확인하는지 파악합니다.
2. 바로 아래 코드 셀을 실행합니다.
3. 출력에서 핵심 값 1-2개만 고릅니다.
4. 그 값을 보고서에 쓸 수 있는 문장으로 바꿉니다.

마지막에는 다음 형태로 정리합니다.

```text
같은 모델과 threshold에서 어떤 오류가 많고, threshold를 바꾸면 어떤 비용이 늘어납니다.
```


## 1. 실습 준비와 실행 범위

### 1-1. Lite와 로컬 실행 기준 고정

이 notebook의 첫 판단은 어떤 실행 환경과 어떤 helper 기준으로 score와 metric을 계산하는지 고정하는 것입니다. JupyterLite에서는 browser-safe `ai_quality.lite` helper를 사용하고, 로컬에서는 저장소의 `packages/ai-quality/src`를 우선 사용합니다.

이 준비 셀은 `utils.py`를 통해 설치와 import만 담당합니다. 모델 평가의 핵심인 데이터 로딩, label 표준화, feature readiness, score 생성, threshold 적용, metric 계산은 뒤 셀에서 Pandas 코드로 드러냅니다.


### 따라하기 안내: 평가 환경 준비

**이 셀에서 할 일**  
모델 평가에 필요한 helper를 불러옵니다.

**실행 후 볼 것**  
positive label, negative label, default threshold를 확인합니다.


In [ ]:
from __future__ import annotations

import utils as ch02

prepared = await ch02.prepare_notebook()
pd = prepared.pandas
aiq_lite = prepared.aiq_lite

environment_summary = pd.DataFrame(
    [
        {"item": "package_module", "value": aiq_lite.__name__, "why_it_matters": "Lite와 로컬에서 같은 score/metric 기준을 제공합니다."},
        {"item": "positive_label", "value": aiq_lite.POSITIVE_LABEL, "why_it_matters": "recall과 precision을 계산할 관심 class입니다."},
        {"item": "negative_label", "value": aiq_lite.NEGATIVE_LABEL, "why_it_matters": "FP를 해석할 비교 class입니다."},
        {"item": "default_threshold", "value": aiq_lite.THRESHOLD, "why_it_matters": "score를 prediction으로 바꾸는 기본 기준입니다."},
    ]
)

display(environment_summary)


이 출력에서 확인할 핵심은 `package_module`과 기본 threshold입니다. `ai_quality.lite`는 Lite에서도 실행 가능한 기준값과 sample loader를 제공하지만, 실제 score 생성과 metric 계산은 아래 Pandas 셀에서 보여줍니다.

`high_risk`는 Positive class이고 `low_risk`는 Negative class입니다. FP는 실제 `low_risk`를 `high_risk`로 잘못 예측한 오탐이고, FN은 실제 `high_risk`를 `low_risk`로 놓친 미탐입니다.

## 2. 평가 데이터 brief와 변환 기준

### 2-1. test 원본 데이터 로딩과 row grain 확인

이 셀의 판단은 모델 평가가 어떤 원본 데이터에서 시작되는지 확인하는 것입니다. metric은 숫자 하나로 보이지만, 그 숫자는 특정 파일, 특정 행 수, 특정 class support, 특정 실행 범위에서만 의미가 있습니다.

`vital_signs_test.csv`의 한 row는 생체신호 기반 위험 알림 시스템의 평가용 sample 하나를 뜻합니다. 이 sample은 실제 의료 판단이 아니라 classification 예제의 입력과 정답 기준입니다.

이 단계에서는 아직 label을 표준화하지 않고 score도 만들지 않습니다. 먼저 source, sample scope, row count, column count, 원본 label 값을 고정해야 뒤에서 metric이 어떤 데이터에서 계산되었는지 설명할 수 있습니다.


### 따라하기 안내: test 데이터 확인

**이 셀에서 할 일**  
최종 평가에 사용할 test 데이터의 기본 상태를 확인합니다.

**실행 후 볼 것**  
row 수, column 수, 원본 label 값을 봅니다.


In [ ]:
feature_columns = aiq_lite.FEATURE_COLUMNS
positive_label = aiq_lite.POSITIVE_LABEL
negative_label = aiq_lite.NEGATIVE_LABEL
operating_threshold = aiq_lite.THRESHOLD
TEST_DATA_PATH = "data/vital_signs_test.csv"

raw_test_dataframe, test_source = aiq_lite.load_csv_or_sample(
    TEST_DATA_PATH,
    aiq_lite.sample_vital_signs(),
    nrows=None,
)

execution_scope = "embedded_fallback_sample" if "embedded" in test_source else (
    "jupyterlite_sample" if len(raw_test_dataframe) <= 600 else "local_full_or_large_sample"
)

raw_provenance = pd.DataFrame(
    [
        {
            "dataset": "vital_signs_test",
            "dataset_role": "selected_test_evaluation_dataset",
            "path": TEST_DATA_PATH,
            "data_source": test_source,
            "execution_scope": execution_scope,
            "row_grain": "one classification sample for risk alert evaluation",
            "row_count": raw_test_dataframe.shape[0],
            "column_count": raw_test_dataframe.shape[1],
        }
    ]
)

raw_label_distribution = (
    raw_test_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_test_dataframe) * 100).round(2))
)

raw_preview_columns = [
    column
    for column in ["patient_id", "timestamp", *feature_columns, "label"]
    if column in raw_test_dataframe.columns
]

raw_data_brief = pd.DataFrame(
    [
        {"check": "source_fixed", "observed": test_source, "qa_use": "metric 계산의 입력 파일 또는 fallback sample 기준을 고정합니다."},
        {"check": "row_grain", "observed": "one classification sample", "qa_use": "metric 분모가 무엇을 세는지 설명합니다."},
        {"check": "raw_label_values", "observed": raw_label_distribution["raw_label"].astype(str).tolist(), "qa_use": "표준화 전 정답 기준 값을 확인합니다."},
        {"check": "score_status", "observed": "not_created_yet", "qa_use": "아직 모델 출력이 아니라 원본 평가 입력만 확인합니다."},
    ]
)


### 출력 확인: `raw_provenance`

**읽는 순서**  
먼저 `source` 또는 `path`를 보고, 그 다음 `execution_scope`나 데이터셋 이름을 확인합니다. 이 출력은 숫자를 해석하기 전에 “어떤 근거를 보고 있는지”를 고정하는 단계입니다.

**해석 기준**  
경로가 prepared artifact이면 준비된 근거를 읽은 것이고, 로컬 파일이면 현재 환경에서 읽은 근거입니다. 실행 범위가 sample이면 뒤의 metric 숫자를 전체 데이터 결과처럼 쓰면 안 됩니다.


In [ ]:
display(raw_provenance)


### 출력 확인: `raw_data_brief`

**읽는 순서**  
행 수, 컬럼 수, 데이터셋 이름, 실행 범위를 순서대로 봅니다. 여기서는 아직 품질 판단을 확정하지 않고 “분석 대상이 맞는지”를 확인합니다.

**해석 기준**  
행 수가 예상보다 작으면 JupyterLite sample 또는 fallback sample일 수 있습니다. 컬럼 수나 데이터셋 이름이 기대와 다르면 뒤의 결측, label, metric 해석이 모두 흔들립니다.


In [ ]:
display(raw_data_brief)


### 출력 확인: `raw_test_dataframe.loc[:, raw_preview_columns].head()`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.


In [ ]:
display(raw_test_dataframe.loc[:, raw_preview_columns].head())


### 출력 확인: `raw_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(raw_label_distribution)


이 출력에서 확인할 핵심은 아직 score나 prediction이 없다는 점입니다. 지금 보이는 것은 평가 입력의 원본 상태이며, 이 단계에서 row 수나 label 값이 기대와 다르면 metric 계산으로 넘어가면 안 됩니다.

`execution_scope`가 `jupyterlite_sample` 또는 `embedded_fallback_sample`이면 숫자는 브라우저 실습용 축약 근거입니다. 보고서에는 전체 로컬 데이터인지 Lite sample인지 반드시 구분해야 합니다.

### 2-2. label 표준화와 class support 확인

이 셀의 판단은 원본 label 값이 평가용 label 계약으로 안전하게 바뀌는지 확인하는 것입니다. label은 정답 기준이므로, score나 prediction을 만들기 전에 `high_risk`와 `low_risk`로 표준화되는 과정을 분리해서 봐야 합니다.

표준화는 원본 dataframe을 직접 덮어쓰지 않고 `copy()` 후 `assign()`으로 진행합니다. 이렇게 해야 원본 label과 표준화 label을 나란히 비교할 수 있고, 나중에 metric 문제가 생겼을 때 label 변환이 원인인지 추적할 수 있습니다.

이 단계의 통과 기준은 허용되지 않은 label이 없고, Positive class와 Negative class가 모두 충분히 존재하는 것입니다. class support가 한쪽으로 치우치면 Precision과 Recall을 해석할 때 제한 사항을 남겨야 합니다.


### 따라하기 안내: label 표준화

**이 셀에서 할 일**  
test label을 평가 기준에 맞춥니다.

**실행 후 볼 것**  
표준화된 label 분포를 확인합니다.

**기록 포인트**  
정답 기준을 고정했다고 적습니다.

**생각해 볼 질문**  
정답지가 흔들리면 metric은 믿을 수 있을까요?


In [ ]:
test_dataframe = raw_test_dataframe.copy().assign(
    label=lambda table: table["label"].map(aiq_lite.normalize_label)
)

label_transition = (
    raw_test_dataframe.loc[:, ["label"]]
    .rename(columns={"label": "raw_label"})
    .assign(standardized_label=test_dataframe["label"])
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_test_dataframe) * 100).round(2))
)

standardized_label_distribution = (
    test_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("standardized_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(test_dataframe) * 100).round(2))
)

class_support = standardized_label_distribution.set_index("standardized_label")["count"].to_dict()
unexpected_label_count = int(~test_dataframe["label"].isin([positive_label, negative_label]).sum())
label_readiness = pd.DataFrame(
    [
        {
            "gate": "label_contract",
            "positive_label": positive_label,
            "negative_label": negative_label,
            "unexpected_label_count": unexpected_label_count,
            "positive_support": int(class_support.get(positive_label, 0)),
            "negative_support": int(class_support.get(negative_label, 0)),
            "status": "pass" if unexpected_label_count == 0 and all(class_support.get(label, 0) > 0 for label in [positive_label, negative_label]) else "hold",
            "qa_judgment": "FP/FN 계산에 필요한 label 기준을 사용할 수 있습니다."
            if unexpected_label_count == 0 and all(class_support.get(label, 0) > 0 for label in [positive_label, negative_label])
            else "metric 계산 전에 label 기준과 class support를 확인해야 합니다.",
        }
    ]
)

label_transformation_audit = pd.DataFrame(
    [
        {"step": "copy_before_mutation", "code_pattern": "raw_test_dataframe.copy()", "qa_reason": "원본 label과 표준화 label을 비교할 수 있게 보존합니다."},
        {"step": "standardize_label", "code_pattern": "Series.map(aiq_lite.normalize_label)", "qa_reason": "정답 기준을 high_risk/low_risk 계약으로 맞춥니다."},
        {"step": "check_class_support", "code_pattern": "value_counts(dropna=False)", "qa_reason": "Precision/Recall 해석에 필요한 class 존재 여부를 확인합니다."},
    ]
)


### 출력 확인: `label_transition`

**읽는 순서**  
원본 label 값과 표준화된 label 값을 나란히 봅니다. 같은 의미의 값이 `high_risk`, `low_risk`로 정리되는지 확인합니다.

**해석 기준**  
표준화 후 label이 비거나 예상 밖 값이 생기면 metric 계산 전에 멈춰야 합니다. label은 모델 평가의 정답 기준이므로 작은 표기 차이도 FP/FN 해석을 바꿀 수 있습니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값이 무엇이었는지도 같이 확인해야 나중에 label mapping 문제를 추적할 수 있습니다.


In [ ]:
display(label_transition)


### 출력 확인: `standardized_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(standardized_label_distribution)


### 출력 확인: `label_readiness`

**읽는 순서**  
invalid label, missing label, positive support, negative support를 차례로 봅니다.

**해석 기준**  
invalid 또는 missing label이 있으면 정답 비교가 흔들립니다. positive support가 부족하면 recall 한두 건 차이도 큰 비율 변화로 보일 수 있습니다.

**보고서 문장 예시**  
“label gate에서 허용 label과 class support를 확인했으며, metric 해석 가능 범위를 판단했습니다.”

**주의할 점**  
label gate 통과는 feature 품질 통과를 의미하지 않습니다. label은 정답 기준만 확인합니다.


In [ ]:
display(label_readiness)


### 출력 확인: `label_transformation_audit`

**읽는 순서**  
원본 label 값과 표준화된 label 값을 나란히 봅니다. 같은 의미의 값이 `high_risk`, `low_risk`로 정리되는지 확인합니다.

**해석 기준**  
표준화 후 label이 비거나 예상 밖 값이 생기면 metric 계산 전에 멈춰야 합니다. label은 모델 평가의 정답 기준이므로 작은 표기 차이도 FP/FN 해석을 바꿀 수 있습니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값이 무엇이었는지도 같이 확인해야 나중에 label mapping 문제를 추적할 수 있습니다.


In [ ]:
display(label_transformation_audit)


이 출력에서 확인할 핵심은 label 변환이 metric 계산 전 단계라는 점입니다. label이 흔들리면 FP와 FN이 모두 흔들리므로, 모델 문제처럼 보이는 지표 변화가 실제로는 정답 기준 문제일 수 있습니다.

이 notebook에서는 label 계약이 통과되어야 score와 prediction을 만들 수 있습니다. 통과하더라도 label 반전 후보 같은 더 깊은 정답 기준 신뢰 문제는 별도 제한 사항으로 남겨야 합니다.

### 2-3. feature readiness와 numeric 변환 전제 확인

이 셀의 판단은 score를 만들 feature가 실제로 존재하고 숫자로 해석 가능한지 확인하는 것입니다. score 산식은 feature 값을 사용하므로, feature 결측이나 숫자 변환 실패는 metric 변화의 원인 후보가 될 수 있습니다.

여기서는 모델 산식으로 바로 넘어가지 않습니다. 먼저 feature 컬럼 존재 여부, 결측 수, 숫자 변환 후 결측 수, 기본 분포를 표로 확인합니다. 이 표는 “metric을 계산해도 되는가”와 “나중에 metric이 흔들릴 때 어떤 feature를 의심할 것인가”를 동시에 정리합니다.

feature readiness는 모델 품질 결론이 아닙니다. 다만 결측이나 범위 이상이 보이면 score 분포와 prediction 분포를 해석할 때 데이터 조건 변화 후보로 남깁니다.


### 따라하기 안내: feature readiness

**이 셀에서 할 일**  
평가에 필요한 feature가 모두 있는지 확인합니다.

**실행 후 볼 것**  
사용 가능한 feature와 누락 feature를 봅니다.

**기록 포인트**  
feature 조건이 유지되는지 적습니다.

**생각해 볼 질문**  
feature가 빠졌다면 모델 문제라고 바로 말할 수 있을까요?


In [ ]:
available_feature_columns = [column for column in feature_columns if column in test_dataframe.columns]
feature_values = test_dataframe.loc[:, available_feature_columns].apply(pd.to_numeric, errors="coerce")

feature_readiness_rows: list[dict[str, object]] = []
for column in feature_columns:
    if column not in test_dataframe.columns:
        feature_readiness_rows.append(
            {
                "feature": column,
                "exists": False,
                "dtype": "missing",
                "missing_count": len(test_dataframe),
                "numeric_na_after_coerce": len(test_dataframe),
                "min": None,
                "median": None,
                "max": None,
                "status": "hold",
            }
        )
        continue
    numeric_values = pd.to_numeric(test_dataframe[column], errors="coerce")
    missing_count = int(test_dataframe[column].isna().sum())
    numeric_na_count = int(numeric_values.isna().sum())
    feature_readiness_rows.append(
        {
            "feature": column,
            "exists": True,
            "dtype": str(test_dataframe[column].dtype),
            "missing_count": missing_count,
            "numeric_na_after_coerce": numeric_na_count,
            "min": round(float(numeric_values.min()), 4) if numeric_values.notna().any() else None,
            "median": round(float(numeric_values.median()), 4) if numeric_values.notna().any() else None,
            "max": round(float(numeric_values.max()), 4) if numeric_values.notna().any() else None,
            "status": "pass" if missing_count == 0 and numeric_na_count == 0 else "review",
        }
    )

feature_readiness = pd.DataFrame(feature_readiness_rows)
feature_contract = pd.DataFrame(
    [
        {
            "gate": "feature_readiness",
            "feature_columns_present": f"{int(feature_readiness['exists'].sum())}/{len(feature_columns)}",
            "features_with_missing": int((feature_readiness["missing_count"] > 0).sum()),
            "features_with_numeric_issue": int((feature_readiness["numeric_na_after_coerce"] > feature_readiness["missing_count"]).sum()),
            "threshold_to_apply_later": operating_threshold,
            "status": "pass" if bool(feature_readiness["exists"].all()) else "hold",
            "qa_judgment": "score 생성에 필요한 feature 컬럼이 준비되었습니다."
            if bool(feature_readiness["exists"].all())
            else "feature 컬럼 누락이 있어 score와 metric 계산을 보류해야 합니다.",
        }
    ]
)

feature_distribution_snapshot = (
    feature_values.describe(percentiles=[0.1, 0.5, 0.9])
    .T
    .loc[:, ["count", "mean", "std", "min", "10%", "50%", "90%", "max"]]
    .round(4)
    .reset_index()
    .rename(columns={"index": "feature"})
)


### 출력 확인: `feature_contract`

**읽는 순서**  
필수 컬럼 또는 feature 목록이 모두 있는지 봅니다. 누락 컬럼이 있으면 어떤 역할의 컬럼인지 확인합니다.

**해석 기준**  
모델 feature가 빠지면 score 계산 조건이 달라지고, label이 빠지면 metric 계산이 불가능합니다. 계약 확인은 성능 계산보다 먼저 보는 gate입니다.


In [ ]:
display(feature_contract)


### 출력 확인: `feature_readiness`

**읽는 순서**  
feature별 missing, invalid, status를 행 단위로 봅니다. 문제가 있는 feature 이름을 먼저 고릅니다.

**해석 기준**  
문제가 있는 feature가 모델 score에 직접 들어가면 metric 변화의 원인 후보가 됩니다. 특히 vital sign feature의 결측과 범위 오류는 score 분포를 움직일 수 있습니다.

**보고서 문장 예시**  
“feature 품질 표에서 문제가 있는 입력 항목을 확인했고, metric 변화의 원인 후보로 남겼습니다.”


In [ ]:
display(feature_readiness)


### 출력 확인: `feature_distribution_snapshot`

**읽는 순서**  
feature별 평균, 최소, 최대 또는 분포 요약을 봅니다. 기준 데이터와 비교되는 경우 변화 방향을 확인합니다.

**해석 기준**  
평균이 움직이면 입력 구성 변화 후보가 됩니다. 하지만 평균만으로 분포 전체를 설명할 수 없으므로 histogram이나 range 결과와 같이 봐야 합니다.


In [ ]:
display(feature_distribution_snapshot)


이 출력에서 확인할 핵심은 score 계산의 입력 전제가 분리되어 있다는 점입니다. `feature_contract`가 통과되어야 뒤의 score가 “같은 feature 목록에서 계산된 값”이라고 말할 수 있습니다.

분포 snapshot은 정상/비정상 판정표가 아니라 비교 기준입니다. 뒤에서 validation baseline과 degraded 데이터를 비교할 때 같은 feature의 결측, 범위, 중앙값 변화가 score와 metric 변화의 원인 후보가 되는지 다시 연결합니다.

## 3. score 생성과 prediction 생성

### 3-1. feature에서 score 만들기

이 셀의 판단은 준비된 feature가 어떻게 `score`로 바뀌는지 확인하는 것입니다. score는 아직 최종 예측이 아니며, Positive class 가능성을 나타내는 모델 출력입니다. 이 notebook에서는 Lite에서도 실행 가능한 고정 scoring rule을 사용합니다.

모델 산식 자체를 성능 개선 대상으로 다루지 않습니다. QA가 확인할 것은 같은 feature 목록과 같은 scoring rule로 score가 생성되었는지, 그리고 score 분포가 이후 prediction과 metric 해석에 어떤 전제가 되는지입니다.

이 셀은 숫자 변환, 결측 대체, clip, component 합산을 transformation audit으로 남깁니다. 나중에 score 분포가 달라졌을 때 어떤 처리 단계가 영향을 줄 수 있는지 설명하기 위해서입니다.


### 따라하기 안내: score 계산

**이 셀에서 할 일**  
각 sample의 `high_risk` score를 계산합니다.

**실행 후 볼 것**  
score 범위와 score 예시를 봅니다.


In [ ]:
score_components = pd.DataFrame(
    {
        "heart_rate_component": ((feature_values["heart_rate"].fillna(75) - 60).clip(lower=0, upper=39) / 39 * 0.70),
        "respiratory_component": ((feature_values["respiratory_rate"].fillna(15) - 12).clip(lower=0, upper=7) / 7 * 0.06),
        "temperature_component": ((feature_values["body_temperature"].fillna(36.7) - 36.0).clip(lower=0, upper=2) / 2 * 0.04),
        "oxygen_component": ((98.5 - feature_values["oxygen_saturation"].fillna(98)).clip(lower=0, upper=4) / 4 * 0.06),
        "systolic_component": ((feature_values["systolic_blood_pressure"].fillna(124) - 115).clip(lower=0, upper=25) / 25 * 0.04),
    }
)

scored_dataframe = test_dataframe.copy().assign(
    score=(0.10 + score_components.sum(axis=1)).clip(lower=0.01, upper=0.99).round(4)
)

score_summary = (
    scored_dataframe["score"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .round(4)
    .rename("score")
    .reset_index()
    .rename(columns={"index": "stat"})
)

component_summary = (
    score_components.agg(["mean", "max"])
    .T
    .round(4)
    .reset_index()
    .rename(columns={"index": "component"})
)
score_transformation_audit = pd.DataFrame(
    [
        {"step": "numeric_feature_values", "operation": "DataFrame.apply(pd.to_numeric, errors='coerce')", "qa_reason": "feature가 score 산식에 들어가기 전에 숫자 값인지 확인합니다."},
        {"step": "missing_default", "operation": "Series.fillna(...) per feature", "qa_reason": "결측이 score에 들어갈 때 어떤 기본값이 쓰였는지 남깁니다."},
        {"step": "bounded_component", "operation": "Series.clip(lower=..., upper=...)", "qa_reason": "극단값이 component를 과도하게 흔들지 않도록 제한한 조건을 기록합니다."},
        {"step": "score", "operation": "0.10 + component sum, clipped to [0.01, 0.99]", "qa_reason": "prediction 이전의 연속값을 만듭니다."},
    ]
)

score_preview_columns = ["patient_id", *feature_columns, "label", "score"]


### 출력 확인: `score_transformation_audit`

**읽는 순서**  
데이터셋, threshold, positive label, feature 조건을 확인합니다.

**해석 기준**  
metric이나 prediction은 조건 없이 해석할 수 없습니다. 같은 score라도 threshold가 다르면 FP/FN이 달라집니다.

**주의할 점**  
조건 확인을 생략하면 서로 다른 실행 결과를 같은 metric처럼 비교하는 실수를 하게 됩니다.


In [ ]:
display(score_transformation_audit)


### 출력 확인: `scored_dataframe.loc[:, score_preview_columns].head()`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.

**주의할 점**  
앞 5개 행만 보고 분포나 성능을 판단하면 안 됩니다. 분포와 metric은 뒤의 별도 출력에서 확인합니다.


In [ ]:
display(scored_dataframe.loc[:, score_preview_columns].head())


### 출력 확인: `score_summary`

**읽는 순서**  
score의 평균, 분위수, 최소/최대를 봅니다. 비교 표라면 baseline과 degraded의 변화 방향을 확인합니다.

**해석 기준**  
score는 threshold 적용 전 모델 출력입니다. score가 이동하면 입력 분포, 모델 버전, preprocessing 변화 후보를 생각할 수 있습니다.

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다. 이 실습에서는 분포 변화와 threshold 영향에 집중합니다.


In [ ]:
display(score_summary)


### 출력 확인: `component_summary`

**읽는 순서**  
score를 구성하는 feature component별 평균과 범위를 봅니다.

**해석 기준**  
어떤 feature component가 score에 크게 기여하는지 보면 입력 변화와 score 변화의 연결 후보를 찾을 수 있습니다.


In [ ]:
display(component_summary)


이 출력에서 확인할 핵심은 `score`가 `label`이나 `prediction`이 아니라는 점입니다. score는 feature에서 생성된 연속값이며, 아직 `high_risk` 또는 `low_risk`라는 최종 class가 아닙니다.

component summary는 어떤 입력 feature가 score에 영향을 주는지 보여줍니다. 점수 분포가 기준과 달라지면 입력 feature 분포 변화 또는 결측/범위 오류를 원인 후보로 볼 수 있습니다.

### 3-2. threshold를 적용해 prediction 만들기

이 셀의 판단은 같은 score라도 threshold에 따라 prediction이 달라진다는 점을 확인하는 것입니다. threshold는 운영 기준이며, 모델이 새로 학습되는 것이 아닙니다.

기본 threshold는 `0.50`입니다. score가 threshold 이상이면 `high_risk`, 그렇지 않으면 `low_risk`로 prediction을 만듭니다. 이 변환을 분리해서 보여야 뒤에서 threshold sweep을 해석할 수 있습니다.


### 따라하기 안내: threshold 적용

**이 셀에서 할 일**  
score를 prediction으로 바꿉니다.

**실행 후 볼 것**  
threshold 값과 prediction 분포를 확인합니다.

**기록 포인트**  
운영 기준으로 prediction이 만들어졌다고 적습니다.

**생각해 볼 질문**  
threshold는 모델 자체인가요, 운영 기준인가요?


In [ ]:
prediction_dataframe = scored_dataframe.assign(
    prediction=lambda table: table["score"].ge(operating_threshold).map(
        {True: positive_label, False: negative_label}
    ),
    is_correct=lambda table: table["label"].eq(table["prediction"]),
)

prediction_distribution = (
    prediction_dataframe["prediction"]
    .value_counts(dropna=False)
    .rename_axis("prediction")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(prediction_dataframe) * 100).round(2))
)

prediction_transformation_audit = pd.DataFrame(
    [
        {"step": "threshold_fixed", "observed": operating_threshold, "qa_reason": "같은 score를 같은 운영 기준으로 class로 변환합니다."},
        {"step": "prediction_rule", "observed": "score >= threshold -> high_risk else low_risk", "qa_reason": "prediction 분포와 label 분포를 분리해서 해석합니다."},
    ]
)

label_prediction_preview = prediction_dataframe.loc[
    :, ["patient_id", "label", "score", "prediction", "is_correct"]
].head(10)


### 출력 확인: `prediction_transformation_audit`

**읽는 순서**  
표의 첫 번째 열부터 읽고, 어떤 항목이 기준값이고 어떤 항목이 관측값인지 구분합니다.

**해석 기준**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다. 핵심 값 1-2개를 골라 다음 셀의 판단과 연결합니다.

**주의할 점**  
표 전체를 그대로 옮기지 말고, 판단에 필요한 값과 그 의미만 보고서에 남깁니다.


In [ ]:
display(prediction_transformation_audit)


### 출력 확인: `label_prediction_preview`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.

**주의할 점**  
앞 5개 행만 보고 분포나 성능을 판단하면 안 됩니다. 분포와 metric은 뒤의 별도 출력에서 확인합니다.


In [ ]:
display(label_prediction_preview)


### 출력 확인: `prediction_distribution`

**읽는 순서**  
`high_risk`와 `low_risk` prediction count와 비율을 봅니다. 비교 표라면 baseline과 degraded의 차이를 확인합니다.

**해석 기준**  
prediction은 score에 threshold를 적용한 결과입니다. prediction 비율이 바뀌면 score 분포 변화인지 threshold 변경인지 구분해야 합니다.

**주의할 점**  
prediction 분포는 정답 label과 비교한 metric이 아닙니다. 성능 판단은 confusion matrix와 metric table에서 확인합니다.


In [ ]:
display(prediction_distribution)


이 출력에서 확인할 핵심은 label 분포와 prediction 분포가 서로 다른 값이라는 점입니다. label은 정답 기준이고 prediction은 score와 threshold로 만든 모델 출력입니다.

`high_risk` prediction 비율이 높아졌다는 관측은 이 단계 이후에야 말할 수 있습니다. 1장의 label 분포와 2장의 prediction 분포를 섞으면 원인 후보를 잘못 좁히게 됩니다.

## 4. Confusion Matrix와 metric 계산

### 4-1. label과 prediction을 비교해 오류 유형 만들기

이 셀의 판단은 prediction이 맞았는지 틀렸는지를 TP, TN, FP, FN으로 나누는 것입니다. Accuracy 하나는 전체 요약이지만, QA 판단에는 FP와 FN의 건수가 반드시 필요합니다.

관심 class는 `high_risk`입니다. FP는 실제 `low_risk`인데 `high_risk`로 예측한 경우이고, FN은 실제 `high_risk`인데 `low_risk`로 예측한 경우입니다.


### 따라하기 안내: confusion matrix

**이 셀에서 할 일**  
TP, FP, FN, TN을 확인합니다.

**실행 후 볼 것**  
FP와 FN 개수를 중심으로 봅니다.

**기록 포인트**  
어떤 오류 유형이 많은지 적습니다.

**생각해 볼 질문**  
이 서비스에서 FP와 FN 중 무엇이 더 위험할까요?


In [ ]:
confusion_table = pd.crosstab(
    prediction_dataframe["label"],
    prediction_dataframe["prediction"],
    rownames=["actual_label"],
    colnames=["prediction"],
    dropna=False,
).reindex(index=[positive_label, negative_label], columns=[positive_label, negative_label], fill_value=0)

tp = int(confusion_table.loc[positive_label, positive_label])
fn = int(confusion_table.loc[positive_label, negative_label])
fp = int(confusion_table.loc[negative_label, positive_label])
tn = int(confusion_table.loc[negative_label, negative_label])

confusion_summary = pd.DataFrame(
    [
        {"type": "TP", "count": tp, "meaning": "actual high_risk, predicted high_risk"},
        {"type": "FP", "count": fp, "meaning": "actual low_risk, predicted high_risk"},
        {"type": "FN", "count": fn, "meaning": "actual high_risk, predicted low_risk"},
        {"type": "TN", "count": tn, "meaning": "actual low_risk, predicted low_risk"},
    ]
)

accuracy = (tp + tn) / len(prediction_dataframe) if len(prediction_dataframe) else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

metric_table = pd.DataFrame(
    [
        {
            "dataset": "vital_signs_test",
            "threshold": operating_threshold,
            "row_count": len(prediction_dataframe),
            "accuracy": round(accuracy, 4),
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    ]
)

metric_input_audit = pd.DataFrame(
    [
        {"input": "label", "source": "standardized test_dataframe.label", "qa_reason": "정답 기준입니다."},
        {"input": "score", "source": "scored_dataframe.score", "qa_reason": "threshold 적용 전 모델 출력입니다."},
        {"input": "prediction", "source": "prediction_dataframe.prediction", "qa_reason": "score와 threshold로 만든 최종 예측입니다."},
        {"input": "threshold", "source": operating_threshold, "qa_reason": "metric 계산 조건으로 기록해야 합니다."},
    ]
)


### 출력 확인: `metric_input_audit`

**읽는 순서**  
데이터셋, threshold, positive label, feature 조건을 확인합니다.

**해석 기준**  
metric이나 prediction은 조건 없이 해석할 수 없습니다. 같은 score라도 threshold가 다르면 FP/FN이 달라집니다.

**보고서 문장 예시**  
“metric 계산 조건으로 데이터셋, threshold, positive class 기준을 확인했습니다.”

**주의할 점**  
조건 확인을 생략하면 서로 다른 실행 결과를 같은 metric처럼 비교하는 실수를 하게 됩니다.


In [ ]:
display(metric_input_audit)


### 출력 확인: `confusion_table`

**읽는 순서**  
TP, FP, FN, TN을 확인합니다. 특히 FP와 FN 중 무엇이 많은지 먼저 봅니다.

**해석 기준**  
FP는 실제 low risk를 high risk로 잘못 알린 경우이고, FN은 실제 high risk를 놓친 경우입니다. 같은 accuracy라도 FP/FN 구성에 따라 운영 리스크가 다릅니다.

**보고서 문장 예시**  
“confusion matrix에서 FP/FN 오류 유형을 확인했고, precision/recall 해석의 근거로 사용했습니다.”

**주의할 점**  
행과 열의 의미를 바꾸어 읽으면 FP와 FN 해석이 반대로 됩니다. actual과 prediction 축을 꼭 확인합니다.


In [ ]:
display(confusion_table)


### 출력 확인: `confusion_summary`

**읽는 순서**  
TP, FP, FN, TN을 확인합니다. 특히 FP와 FN 중 무엇이 많은지 먼저 봅니다.

**해석 기준**  
FP는 실제 low risk를 high risk로 잘못 알린 경우이고, FN은 실제 high risk를 놓친 경우입니다. 같은 accuracy라도 FP/FN 구성에 따라 운영 리스크가 다릅니다.

**보고서 문장 예시**  
“confusion matrix에서 FP/FN 오류 유형을 확인했고, precision/recall 해석의 근거로 사용했습니다.”

**주의할 점**  
행과 열의 의미를 바꾸어 읽으면 FP와 FN 해석이 반대로 됩니다. actual과 prediction 축을 꼭 확인합니다.


In [ ]:
display(confusion_summary)


### 출력 확인: `metric_table`

**읽는 순서**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다. 하나만 고르지 말고 FP/FN과 연결합니다.

**해석 기준**  
precision은 FP 영향을 크게 받고, recall은 FN 영향을 크게 받습니다. 이 실습에서는 accuracy보다 어떤 오류가 늘었는지가 중요합니다.

**보고서 문장 예시**  
“metric 표에서 precision/recall과 FP/FN을 함께 확인해 모델 품질 판단 근거로 사용했습니다.”

**주의할 점**  
metric 값만 복사하지 않습니다. 데이터셋, threshold, positive label 조건을 함께 적어야 합니다.


In [ ]:
display(metric_table)


이 출력에서 확인할 핵심은 오류 유형입니다. Precision은 FP의 영향을 받고 Recall은 FN의 영향을 받습니다. 따라서 Accuracy가 높거나 낮다는 문장만으로는 운영 리스크를 설명할 수 없습니다.

보고서에는 metric 값과 함께 threshold, row count, FP, FN을 같이 남깁니다. 그래야 reviewer가 같은 데이터와 같은 기준에서 계산한 값인지 추적할 수 있습니다.

## 5. 기준 validation과 품질 저하 validation 비교

### 5-1. validation 원본 데이터 두 개를 metric 전에 먼저 비교

이 셀의 판단은 metric delta를 보기 전에 baseline과 degraded 데이터가 어떤 입력 조건을 갖는지 확인하는 것입니다. 같은 모델과 같은 threshold를 적용하더라도 데이터 조건이 달라지면 score, prediction, metric이 달라질 수 있습니다.

`valid_baseline`은 기준 validation 조건이고, `valid_degraded`는 현재 운영 입력에서 나타날 수 있는 결측, 범위 오류, 라벨 기준 흔들림을 교육용으로 재현한 비교 데이터입니다. 이 비교는 실제 운영 로그 자체가 아니라, 같은 평가 조건에서 데이터 품질 후보를 분리해 보기 위한 validation evidence입니다.

이 단계에서는 아직 score를 만들지 않습니다. 먼저 source, row count, raw label 분포, preview를 나란히 확인해 “무엇이 비교되고 있는가”를 고정합니다.


### 따라하기 안내: validation 비교 준비

**이 셀에서 할 일**  
baseline과 degraded validation 비교를 준비합니다.

**실행 후 볼 것**  
비교할 데이터셋 이름을 확인합니다.

**기록 포인트**  
모델과 threshold를 고정한 비교라고 적습니다.

**생각해 볼 질문**  
모델이 같다면 metric 변화 후보는 어디에서 찾을까요?


In [ ]:
VALIDATION_DATASETS = [
    ("valid_baseline", "data/vital_signs_valid_baseline.csv"),
    ("valid_degraded", "data/vital_signs_valid_degraded.csv"),
]

raw_validation_frames: dict[str, pd.DataFrame] = {}
validation_source_rows: list[dict[str, object]] = []
validation_preview_frames: list[pd.DataFrame] = []
validation_raw_label_frames: list[pd.DataFrame] = []

for dataset_name, dataset_path in VALIDATION_DATASETS:
    raw_frame, source = aiq_lite.load_csv_or_sample(dataset_path, aiq_lite.sample_vital_signs(), nrows=None)
    raw_validation_frames[dataset_name] = raw_frame
    scope = "embedded_fallback_sample" if "embedded" in source else (
        "jupyterlite_sample" if len(raw_frame) <= 600 else "local_full_or_large_sample"
    )
    validation_source_rows.append(
        {
            "dataset": dataset_name,
            "path": dataset_path,
            "data_source": source,
            "execution_scope": scope,
            "row_grain": "one validation classification sample",
            "row_count": raw_frame.shape[0],
            "column_count": raw_frame.shape[1],
        }
    )
    validation_preview_frames.append(
        raw_frame.loc[:, raw_preview_columns]
        .head(3)
        .assign(dataset=dataset_name)
        .loc[:, ["dataset", *raw_preview_columns]]
    )
    validation_raw_label_frames.append(
        raw_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("raw_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(raw_frame): (table["count"] / denominator * 100).round(2),
        )
    )

validation_source_table = pd.DataFrame(validation_source_rows)
validation_raw_preview = pd.concat(validation_preview_frames, ignore_index=True)
validation_raw_label_distribution = pd.concat(validation_raw_label_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "count", "ratio_pct"]
]

validation_comparison_brief = pd.DataFrame(
    [
        {"check": "same_row_grain", "observed": "one validation classification sample", "qa_use": "두 데이터셋의 metric 분모가 같은 의미인지 확인합니다."},
        {"check": "score_status", "observed": "not_created_yet", "qa_use": "metric 전에 데이터 조건 차이를 먼저 확인합니다."},
        {"check": "comparison_limit", "observed": "validation evidence, not live operation log", "qa_use": "운영 원인 확정이 아니라 후보 강화 근거로 사용합니다."},
    ]
)


### 출력 확인: `validation_source_table`

**읽는 순서**  
먼저 `source` 또는 `path`를 보고, 그 다음 `execution_scope`나 데이터셋 이름을 확인합니다. 이 출력은 숫자를 해석하기 전에 “어떤 근거를 보고 있는지”를 고정하는 단계입니다.

**해석 기준**  
경로가 prepared artifact이면 준비된 근거를 읽은 것이고, 로컬 파일이면 현재 환경에서 읽은 근거입니다. 실행 범위가 sample이면 뒤의 metric 숫자를 전체 데이터 결과처럼 쓰면 안 됩니다.


In [ ]:
display(validation_source_table)


### 출력 확인: `validation_comparison_brief`

**읽는 순서**  
표의 첫 번째 열부터 읽고, 어떤 항목이 기준값이고 어떤 항목이 관측값인지 구분합니다.

**해석 기준**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다. 핵심 값 1-2개를 골라 다음 셀의 판단과 연결합니다.


In [ ]:
display(validation_comparison_brief)


### 출력 확인: `validation_raw_preview`

**읽는 순서**  
왼쪽부터 컬럼 이름을 보고, 한 행이 무엇을 의미하는지 확인합니다. 그 다음 feature, label, score, prediction 같은 주요 컬럼이 기대한 위치에 있는지 봅니다.

**해석 기준**  
Preview는 전체 통계가 아니라 샘플 행입니다. 값의 모양, 단위, 컬럼 이름, 변환 후 새 컬럼이 생겼는지를 확인하는 용도입니다.


In [ ]:
display(validation_raw_preview)


### 출력 확인: `validation_raw_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(validation_raw_label_distribution)


이 출력에서 확인할 핵심은 baseline과 degraded가 같은 의미의 validation sample을 비교한다는 점입니다. row grain이 다르면 metric delta는 데이터 조건 변화가 아니라 비교 대상 차이를 반영할 수 있습니다.

raw label 분포는 score 이전의 정답 기준 상태입니다. 허용 label 값만 보인다고 해서 정답 기준이 완전히 신뢰된다는 뜻은 아니지만, 최소한 metric 계산 형식은 유지되는지 확인할 수 있습니다.

### 5-2. validation label 표준화와 feature 품질 신호 비교

이 셀의 판단은 degraded 데이터가 metric 계산 전에 어떤 품질 신호를 갖는지 확인하는 것입니다. metric delta를 먼저 보면 “모델이 나빠졌다”로 단정하기 쉽지만, 결측, 범위 오류, label 분포 차이가 먼저 보이면 데이터 조건 변화 후보를 남길 수 있습니다.

두 데이터셋 모두 같은 label 표준화와 같은 feature readiness 기준을 적용합니다. baseline에는 없는 결측이나 범위 이상이 degraded에 나타나면, 뒤에서 score 분포와 FP/FN 변화의 원인 후보로 연결합니다.

이 비교가 증명하는 것은 “데이터 조건 차이가 관측되었다”는 점입니다. 이 비교만으로 모델 자체 결함이나 실제 운영 장애를 확정하지 않고, score/prediction/metric 변화와 함께 해석합니다.


### 따라하기 안내: validation 데이터 로딩

**이 셀에서 할 일**  
두 validation 데이터를 같은 방식으로 읽습니다.

**실행 후 볼 것**  
row 수와 label 분포를 비교합니다.

**기록 포인트**  
degraded validation은 교육용 비교 artifact라고 적습니다.

**생각해 볼 질문**  
이 결과로 운영 root cause를 확정할 수 있을까요?


In [ ]:
validation_frames: dict[str, pd.DataFrame] = {}
validation_label_transition_frames: list[pd.DataFrame] = []
validation_label_distribution_frames: list[pd.DataFrame] = []
feature_quality_rows: list[dict[str, object]] = []
range_quality_rows: list[dict[str, object]] = []

for dataset_name, raw_frame in raw_validation_frames.items():
    standardized_frame = raw_frame.copy().assign(
        label=lambda table: table["label"].map(aiq_lite.normalize_label),
        dataset=dataset_name,
    )
    validation_frames[dataset_name] = standardized_frame

    validation_label_transition_frames.append(
        raw_frame.loc[:, ["label"]]
        .rename(columns={"label": "raw_label"})
        .assign(standardized_label=standardized_frame["label"], dataset=dataset_name)
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )
    validation_label_distribution_frames.append(
        standardized_frame["label"]
        .value_counts(dropna=False)
        .rename_axis("standardized_label")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(standardized_frame): (table["count"] / denominator * 100).round(2),
        )
    )

    numeric_features = standardized_frame.loc[:, feature_columns].apply(pd.to_numeric, errors="coerce")
    rows_with_any_feature_missing = int(standardized_frame.loc[:, feature_columns].isna().any(axis=1).sum())
    feature_quality_rows.append(
        {
            "dataset": dataset_name,
            "row_count": len(standardized_frame),
            "rows_with_any_feature_missing": rows_with_any_feature_missing,
            "missing_row_pct": round(rows_with_any_feature_missing / len(standardized_frame) * 100, 2),
            "heart_rate_median": round(float(numeric_features["heart_rate"].median()), 4),
            "oxygen_saturation_min": round(float(numeric_features["oxygen_saturation"].min()), 4),
            "oxygen_saturation_missing": int(standardized_frame["oxygen_saturation"].isna().sum()),
        }
    )

    for column, (minimum, maximum) in aiq_lite.VALID_RANGES.items():
        if column not in standardized_frame.columns:
            continue
        values = pd.to_numeric(standardized_frame[column], errors="coerce")
        invalid_count = int((values.notna() & ((values < minimum) | (values > maximum))).sum())
        range_quality_rows.append(
            {
                "dataset": dataset_name,
                "feature": column,
                "valid_min": minimum,
                "valid_max": maximum,
                "invalid_count": invalid_count,
                "invalid_pct": round(invalid_count / len(standardized_frame) * 100, 2),
            }
        )

validation_label_transition = pd.concat(validation_label_transition_frames, ignore_index=True).loc[
    :, ["dataset", "raw_label", "standardized_label", "count"]
]
validation_label_distribution = pd.concat(validation_label_distribution_frames, ignore_index=True).loc[
    :, ["dataset", "standardized_label", "count", "ratio_pct"]
]
validation_feature_quality = pd.DataFrame(feature_quality_rows)
validation_range_quality = pd.DataFrame(range_quality_rows).sort_values(
    ["invalid_count", "dataset", "feature"], ascending=[False, True, True]
)

quality_delta_columns = ["rows_with_any_feature_missing", "missing_row_pct", "heart_rate_median", "oxygen_saturation_min", "oxygen_saturation_missing"]
quality_by_dataset = validation_feature_quality.set_index("dataset")
validation_quality_delta = (
    (quality_by_dataset.loc["valid_degraded", quality_delta_columns] - quality_by_dataset.loc["valid_baseline", quality_delta_columns])
    .to_frame()
    .T
    .round(4)
    .assign(comparison="valid_degraded_minus_baseline")
    .loc[:, ["comparison", *quality_delta_columns]]
)

validation_data_risk = pd.DataFrame(
    [
        {
            "risk_candidate": "feature_missing_or_range_shift",
            "observed": "degraded feature quality is compared before scoring",
            "can_explain": "score distribution, prediction distribution, FP/FN movement",
            "cannot_prove": "model defect by itself",
            "next_evidence": "score and metric comparison under same threshold",
        }
    ]
)


### 출력 확인: `validation_label_transition`

**읽는 순서**  
원본 label 값과 표준화된 label 값을 나란히 봅니다. 같은 의미의 값이 `high_risk`, `low_risk`로 정리되는지 확인합니다.

**해석 기준**  
표준화 후 label이 비거나 예상 밖 값이 생기면 metric 계산 전에 멈춰야 합니다. label은 모델 평가의 정답 기준이므로 작은 표기 차이도 FP/FN 해석을 바꿀 수 있습니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값이 무엇이었는지도 같이 확인해야 나중에 label mapping 문제를 추적할 수 있습니다.


In [ ]:
display(validation_label_transition)


### 출력 확인: `validation_label_distribution`

**읽는 순서**  
`high_risk`와 `low_risk`의 count와 ratio를 봅니다. `high_risk`는 positive class이므로 표본 수가 recall 해석의 기본 조건입니다.

**해석 기준**  
두 class가 모두 존재하면 metric 계산은 가능합니다. 한 class가 너무 적으면 accuracy보다 precision/recall과 support를 함께 읽어야 합니다.

**주의할 점**  
class 비율이 균형적이라고 해서 데이터 품질이 모두 좋다는 뜻은 아닙니다. 결측, 범위, label validity는 별도로 봅니다.


In [ ]:
display(validation_label_distribution)


### 출력 확인: `validation_feature_quality`

**읽는 순서**  
feature별 missing, invalid, status를 행 단위로 봅니다. 문제가 있는 feature 이름을 먼저 고릅니다.

**해석 기준**  
문제가 있는 feature가 모델 score에 직접 들어가면 metric 변화의 원인 후보가 됩니다. 특히 vital sign feature의 결측과 범위 오류는 score 분포를 움직일 수 있습니다.

**보고서 문장 예시**  
“feature 품질 표에서 문제가 있는 입력 항목을 확인했고, metric 변화의 원인 후보로 남겼습니다.”

**주의할 점**  
feature 품질 문제를 바로 모델 결함으로 쓰지 않습니다. 먼저 데이터 수집, 전처리, 입력 계약 문제를 후보로 둡니다.


In [ ]:
display(validation_feature_quality)


### 출력 확인: `validation_quality_delta`

**읽는 순서**  
baseline 대비 증가/감소 방향을 봅니다. 양수와 음수가 각각 좋은 변화인지 나쁜 변화인지 항목별로 해석합니다.

**해석 기준**  
delta는 원인을 확정하지 않고 변화가 발생한 위치를 알려 줍니다. metric delta는 데이터 품질 delta, score delta와 함께 봐야 합니다.

**보고서 문장 예시**  
“baseline 대비 변화량에서 어떤 품질 신호와 metric이 함께 움직였는지 확인했습니다.”

**주의할 점**  
모든 delta의 부호가 같은 의미는 아닙니다. 예를 들어 recall delta 감소와 FP delta 증가는 서로 다른 오류 변화를 뜻합니다.


In [ ]:
display(validation_quality_delta)


### 출력 확인: `validation_range_quality.head(12)`

**읽는 순서**  
feature별 최소/최대와 invalid count를 봅니다. 도메인상 가능한 범위를 벗어난 값이 있는지 확인합니다.

**해석 기준**  
범위 오류는 API가 정상 응답해도 score를 비정상적으로 움직일 수 있습니다. 특히 산소포화도, 체온, 혈압 같은 feature는 값의 의미가 중요합니다.

**보고서 문장 예시**  
“range gate에서 허용 범위를 벗어난 feature를 확인했고, score/metric 해석의 제한 사항으로 기록했습니다.”

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다. 먼저 수집 오류인지, 단위 문제인지, 실제 rare case인지 확인해야 합니다.


In [ ]:
display(validation_range_quality.head(12))


### 출력 확인: `validation_data_risk`

**읽는 순서**  
표의 첫 번째 열부터 읽고, 어떤 항목이 기준값이고 어떤 항목이 관측값인지 구분합니다.

**해석 기준**  
이 출력은 바로 앞 코드 셀이 만든 단일 근거입니다. 핵심 값 1-2개를 골라 다음 셀의 판단과 연결합니다.

**보고서 문장 예시**  
“이 출력에서 핵심 관측값을 확인했고, 다음 단계 판단의 입력으로 사용했습니다.”


In [ ]:
display(validation_data_risk)


이 출력에서 확인할 핵심은 metric delta보다 앞선 데이터 차이입니다. degraded 데이터에서 결측 행, 범위 이상, 특정 feature 분포 변화가 보이면 score와 prediction 변화의 원인 후보가 강화됩니다.

다만 이 표는 원인을 확정하지 않습니다. 데이터 조건 차이가 있다는 근거를 만든 뒤, 같은 score 산식과 같은 threshold에서 실제 metric 변화가 같은 방향으로 나타나는지 확인해야 합니다.

### 5-3. 같은 scoring rule과 threshold로 validation metric 비교

이 셀의 판단은 모델 자체를 바꾸지 않았을 때 데이터 조건 변화가 score, prediction, metric에 어떤 차이를 만드는지 확인하는 것입니다. 앞 셀에서 데이터 차이를 먼저 확인했으므로, 이제 metric delta를 데이터 조건 후보와 연결할 수 있습니다.

두 데이터셋에는 같은 feature 목록, 같은 score 산식, 같은 threshold를 적용합니다. 이 조건을 고정해야 metric 변화가 모델 재학습이나 threshold 변경 때문인지 혼동하지 않습니다.

출력은 score 분포, prediction 분포, metric delta를 순서대로 보여줍니다. score와 prediction이 먼저 움직이고 FP/FN이 함께 변하면 데이터 조건 변화 후보를 다음 분석으로 넘길 수 있습니다.


### 따라하기 안내: 같은 기준 적용

**이 셀에서 할 일**  
두 데이터셋에 같은 score 계산과 threshold를 적용합니다.

**실행 후 볼 것**  
metric 변화와 오류 유형 변화를 봅니다.


In [ ]:
def add_scores_and_predictions(frame: pd.DataFrame, threshold: float) -> pd.DataFrame:
    values = frame.loc[:, feature_columns].apply(pd.to_numeric, errors="coerce")
    components = pd.DataFrame(
        {
            "heart_rate_component": ((values["heart_rate"].fillna(75) - 60).clip(lower=0, upper=39) / 39 * 0.70),
            "respiratory_component": ((values["respiratory_rate"].fillna(15) - 12).clip(lower=0, upper=7) / 7 * 0.06),
            "temperature_component": ((values["body_temperature"].fillna(36.7) - 36.0).clip(lower=0, upper=2) / 2 * 0.04),
            "oxygen_component": ((98.5 - values["oxygen_saturation"].fillna(98)).clip(lower=0, upper=4) / 4 * 0.06),
            "systolic_component": ((values["systolic_blood_pressure"].fillna(124) - 115).clip(lower=0, upper=25) / 25 * 0.04),
        }
    )
    return frame.copy().assign(
        score=(0.10 + components.sum(axis=1)).clip(lower=0.01, upper=0.99).round(4),
        prediction=lambda table: table["score"].ge(threshold).map({True: positive_label, False: negative_label}),
    )


def metric_summary(frame: pd.DataFrame, dataset_name: str, threshold: float) -> dict[str, object]:
    table = pd.crosstab(frame["label"], frame["prediction"]).reindex(
        index=[positive_label, negative_label],
        columns=[positive_label, negative_label],
        fill_value=0,
    )
    tp_value = int(table.loc[positive_label, positive_label])
    fn_value = int(table.loc[positive_label, negative_label])
    fp_value = int(table.loc[negative_label, positive_label])
    tn_value = int(table.loc[negative_label, negative_label])
    precision_value = tp_value / (tp_value + fp_value) if (tp_value + fp_value) else 0.0
    recall_value = tp_value / (tp_value + fn_value) if (tp_value + fn_value) else 0.0
    accuracy_value = (tp_value + tn_value) / len(frame) if len(frame) else 0.0
    f1_value = 2 * precision_value * recall_value / (precision_value + recall_value) if (precision_value + recall_value) else 0.0
    return {
        "dataset": dataset_name,
        "threshold": threshold,
        "row_count": len(frame),
        "accuracy": round(accuracy_value, 4),
        "precision": round(precision_value, 4),
        "recall": round(recall_value, 4),
        "f1": round(f1_value, 4),
        "TP": tp_value,
        "FP": fp_value,
        "FN": fn_value,
        "TN": tn_value,
    }

valid_baseline_scored = add_scores_and_predictions(validation_frames["valid_baseline"], operating_threshold)
valid_degraded_scored = add_scores_and_predictions(validation_frames["valid_degraded"], operating_threshold)

validation_scored_frames = {
    "valid_baseline": valid_baseline_scored,
    "valid_degraded": valid_degraded_scored,
}

score_distribution_by_dataset = pd.concat(
    [
        frame["score"]
        .describe(percentiles=[0.1, 0.5, 0.9])
        .rename(dataset_name)
        .to_frame()
        .T
        .assign(dataset=dataset_name)
        for dataset_name, frame in validation_scored_frames.items()
    ],
    ignore_index=True,
).loc[:, ["dataset", "count", "mean", "std", "min", "10%", "50%", "90%", "max"]].round(4)

prediction_distribution_by_dataset = pd.concat(
    [
        frame["prediction"]
        .value_counts(dropna=False)
        .rename_axis("prediction")
        .reset_index(name="count")
        .assign(
            dataset=dataset_name,
            ratio_pct=lambda table, denominator=len(frame): (table["count"] / denominator * 100).round(2),
        )
        for dataset_name, frame in validation_scored_frames.items()
    ],
    ignore_index=True,
).loc[:, ["dataset", "prediction", "count", "ratio_pct"]]

validation_metrics = pd.DataFrame(
    [
        metric_summary(valid_baseline_scored, "valid_baseline", operating_threshold),
        metric_summary(valid_degraded_scored, "valid_degraded", operating_threshold),
    ]
)
metric_columns = ["accuracy", "precision", "recall", "f1", "FP", "FN"]
metric_by_dataset = validation_metrics.set_index("dataset")
metric_delta = (
    (metric_by_dataset.loc["valid_degraded", metric_columns] - metric_by_dataset.loc["valid_baseline", metric_columns])
    .to_frame()
    .T
    .round(4)
    .assign(comparison="valid_degraded_minus_baseline")
    .loc[:, ["comparison", *metric_columns]]
)

validation_metric_audit = pd.DataFrame(
    [
        {"condition": "same_feature_columns", "observed": f"{len(feature_columns)} features", "qa_reason": "입력 feature 조건을 고정합니다."},
        {"condition": "same_scoring_rule", "observed": "browser_safe_score_components_v1", "qa_reason": "모델 조건 변화와 데이터 조건 변화를 섞지 않습니다."},
        {"condition": "same_threshold", "observed": operating_threshold, "qa_reason": "threshold 변경 효과를 제외합니다."},
    ]
)


### 출력 확인: `validation_metric_audit`

**읽는 순서**  
데이터셋, threshold, positive label, feature 조건을 확인합니다.

**해석 기준**  
metric이나 prediction은 조건 없이 해석할 수 없습니다. 같은 score라도 threshold가 다르면 FP/FN이 달라집니다.

**보고서 문장 예시**  
“metric 계산 조건으로 데이터셋, threshold, positive class 기준을 확인했습니다.”

**주의할 점**  
조건 확인을 생략하면 서로 다른 실행 결과를 같은 metric처럼 비교하는 실수를 하게 됩니다.


In [ ]:
display(validation_metric_audit)


### 출력 확인: `score_distribution_by_dataset`

**읽는 순서**  
score의 평균, 분위수, 최소/최대를 봅니다. 비교 표라면 baseline과 degraded의 변화 방향을 확인합니다.

**해석 기준**  
score는 threshold 적용 전 모델 출력입니다. score가 이동하면 입력 분포, 모델 버전, preprocessing 변화 후보를 생각할 수 있습니다.

**주의할 점**  
score를 실제 확률이라고 단정하지 않습니다. 이 실습에서는 분포 변화와 threshold 영향에 집중합니다.


In [ ]:
display(score_distribution_by_dataset)


### 출력 확인: `prediction_distribution_by_dataset`

**읽는 순서**  
`high_risk`와 `low_risk` prediction count와 비율을 봅니다. 비교 표라면 baseline과 degraded의 차이를 확인합니다.

**해석 기준**  
prediction은 score에 threshold를 적용한 결과입니다. prediction 비율이 바뀌면 score 분포 변화인지 threshold 변경인지 구분해야 합니다.

**주의할 점**  
prediction 분포는 정답 label과 비교한 metric이 아닙니다. 성능 판단은 confusion matrix와 metric table에서 확인합니다.


In [ ]:
display(prediction_distribution_by_dataset)


### 출력 확인: `validation_metrics`

**읽는 순서**  
accuracy, precision, recall, F1, AUROC/PR-AUC를 봅니다. 하나만 고르지 말고 FP/FN과 연결합니다.

**해석 기준**  
precision은 FP 영향을 크게 받고, recall은 FN 영향을 크게 받습니다. 이 실습에서는 accuracy보다 어떤 오류가 늘었는지가 중요합니다.

**보고서 문장 예시**  
“metric 표에서 precision/recall과 FP/FN을 함께 확인해 모델 품질 판단 근거로 사용했습니다.”

**주의할 점**  
metric 값만 복사하지 않습니다. 데이터셋, threshold, positive label 조건을 함께 적어야 합니다.


In [ ]:
display(validation_metrics)


### 출력 확인: `metric_delta`

**읽는 순서**  
baseline 대비 증가/감소 방향을 봅니다. 양수와 음수가 각각 좋은 변화인지 나쁜 변화인지 항목별로 해석합니다.

**해석 기준**  
delta는 원인을 확정하지 않고 변화가 발생한 위치를 알려 줍니다. metric delta는 데이터 품질 delta, score delta와 함께 봐야 합니다.

**보고서 문장 예시**  
“baseline 대비 변화량에서 어떤 품질 신호와 metric이 함께 움직였는지 확인했습니다.”

**주의할 점**  
모든 delta의 부호가 같은 의미는 아닙니다. 예를 들어 recall delta 감소와 FP delta 증가는 서로 다른 오류 변화를 뜻합니다.


In [ ]:
display(metric_delta)


이 출력에서 확인할 핵심은 같은 scoring rule과 같은 threshold에서 어떤 metric이 변했는가입니다. Precision이 낮아지거나 FP가 증가하면 오탐 증가 후보를 남기고, Recall이 낮아지거나 FN이 증가하면 미탐 증가 후보를 남깁니다.

이 비교만으로 모델 자체 결함을 확정하지 않습니다. 앞에서 확인한 데이터 조건 차이가 score와 prediction 분포를 거쳐 metric 변화와 함께 나타나는지 확인한 것이므로, 다음 notebook에서는 데이터 품질 신호와 metric 변화를 더 직접 연결해야 합니다.

## 6. threshold sweep과 오류 trade-off

### 6-1. 같은 score에 threshold만 바꿔 적용

이 셀의 판단은 threshold 변경이 모델 재학습이 아니라 운영 기준 변경이라는 점을 확인하는 것입니다. 같은 score라도 threshold를 낮추면 `high_risk` prediction이 늘고, threshold를 높이면 줄어듭니다.

threshold sweep은 지표를 좋게 보이게 만드는 작업이 아닙니다. FP와 FN 중 어떤 오류가 더 부담인지 판단하기 위한 trade-off evidence입니다.


### 따라하기 안내: threshold 후보 비교

**이 셀에서 할 일**  
threshold를 바꿨을 때 FP/FN이 어떻게 움직이는지 봅니다.

**실행 후 볼 것**  
precision, recall, FP, FN 변화를 함께 봅니다.

**기록 포인트**  
threshold 변경은 tradeoff라고 적습니다.

**생각해 볼 질문**  
threshold를 낮추면 어떤 장점과 비용이 생기나요?


In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows: list[dict[str, object]] = []

for threshold in thresholds:
    threshold_frame = scored_dataframe.assign(
        prediction=lambda table, value=threshold: table["score"].ge(value).map(
            {True: positive_label, False: negative_label}
        )
    )
    row = metric_summary(threshold_frame, "vital_signs_test", threshold)
    threshold_rows.append(
        {
            "threshold": threshold,
            "predicted_high_risk": int((threshold_frame["prediction"] == positive_label).sum()),
            "predicted_high_risk_pct": round((threshold_frame["prediction"] == positive_label).mean() * 100, 2),
            "precision": row["precision"],
            "recall": row["recall"],
            "FP": row["FP"],
            "FN": row["FN"],
        }
    )

threshold_sweep = pd.DataFrame(threshold_rows)
display(threshold_sweep)


이 출력에서 확인할 핵심은 threshold가 오류 유형의 균형을 바꾼다는 점입니다. 낮은 threshold는 Recall을 높일 수 있지만 FP를 늘릴 수 있고, 높은 threshold는 FP를 줄일 수 있지만 FN을 늘릴 수 있습니다.

보고서에는 선택한 threshold와 함께 FP/FN trade-off를 씁니다. “threshold를 낮추면 Recall이 올라간다”에서 멈추지 않고, 오탐 증가가 운영 부담으로 이어지는지까지 적어야 합니다.

## 7. Evidence packet과 보고 문장

### 7-1. 모델 평가 결과를 QA handoff로 조립

마지막 판단은 앞의 출력들을 다음 notebook과 보고서로 넘길 수 있는 evidence packet으로 정리하는 것입니다. 이 packet은 최종 release 승인 문서가 아니라, 데이터 품질과 metric 연결 분석으로 넘기는 모델 평가 증거입니다.

`ready_for_metric_connection`은 데이터 품질 원인 후보를 더 확인할 준비가 되었다는 뜻입니다. 모델 성능이 충분하다는 승인 문구로 쓰면 안 됩니다.


### 따라하기 안내: QA 문장 정리

**이 셀에서 할 일**  
평가 결과를 보고서 문장으로 정리합니다.

**실행 후 볼 것**  
문장에 모델, 데이터셋, threshold 조건이 들어 있는지 봅니다.


In [ ]:
baseline_metric = metric_table.iloc[0].to_dict()
validation_delta = metric_delta.iloc[0].to_dict()
quality_delta = validation_quality_delta.iloc[0].to_dict()

if validation_delta.get("FP", 0) > 0 or validation_delta.get("FN", 0) > 0:
    next_action = "데이터 품질 신호와 FP/FN 변화를 연결해 원인 후보를 좁힙니다."
    decision = "ready_for_metric_connection"
else:
    next_action = "score와 prediction 분포를 확인해 threshold 민감도를 기록합니다."
    decision = "ready_for_metric_connection"

chapter_02_model_metric_packet = pd.DataFrame(
    [
        {
            "evidence": "test_data_brief",
            "observed": (
                f"source={test_source}, scope={execution_scope}, rows={len(test_dataframe)}, "
                f"feature_columns_present={int(feature_readiness['exists'].sum())}/{len(feature_columns)}"
            ),
            "qa_judgment": "metric 숫자보다 먼저 데이터 source, row grain, label/feature readiness를 고정했습니다.",
            "owner": "QA/Data Engineering",
            "next_action": "보고서에 Lite sample 여부와 데이터 범위를 함께 적습니다.",
        },
        {
            "evidence": "test_metric",
            "observed": (
                f"threshold={baseline_metric['threshold']}, "
                f"precision={baseline_metric['precision']}, recall={baseline_metric['recall']}, "
                f"FP={baseline_metric['FP']}, FN={baseline_metric['FN']}"
            ),
            "qa_judgment": "Accuracy 단독이 아니라 FP/FN과 Precision/Recall을 함께 해석합니다.",
            "owner": "QA",
            "next_action": "validation 비교와 threshold sweep을 함께 첨부합니다.",
        },
        {
            "evidence": "validation_data_quality_delta",
            "observed": (
                f"missing_row_pct_delta={quality_delta.get('missing_row_pct')}, "
                f"oxygen_saturation_min_delta={quality_delta.get('oxygen_saturation_min')}"
            ),
            "qa_judgment": "metric delta 전에 degraded 데이터의 입력 조건 차이를 확인했습니다.",
            "owner": "QA/Data Engineering",
            "next_action": "데이터 조건 변화가 score와 prediction 변화로 이어지는지 확인합니다.",
        },
        {
            "evidence": "validation_degradation_delta",
            "observed": (
                f"precision_delta={validation_delta.get('precision')}, "
                f"recall_delta={validation_delta.get('recall')}, "
                f"FP_delta={validation_delta.get('FP')}, FN_delta={validation_delta.get('FN')}"
            ),
            "qa_judgment": "같은 score 기준에서 데이터 조건 변화와 metric 변화가 함께 관측되는지 확인합니다.",
            "owner": "QA/Data Engineering",
            "next_action": next_action,
        },
        {
            "evidence": "threshold_sweep",
            "observed": f"checked_thresholds={thresholds}",
            "qa_judgment": "threshold 변경은 모델 개선이 아니라 FP/FN trade-off 검토입니다.",
            "owner": "QA/Product Owner",
            "next_action": "오탐/미탐 중 더 큰 운영 리스크를 release gate에서 명시합니다.",
        },
    ]
)

chapter_02_handoff = pd.DataFrame(
    [
        {
            "chapter": "02_model_quality",
            "decision": decision,
            "decision_reason": "data brief, score, prediction, confusion matrix, metric, threshold trade-off를 같은 기준에서 확인했습니다.",
            "open_candidates": "input_quality_shift, threshold_tradeoff, label_basis_review",
            "next_chapter_question": "데이터 품질 신호가 score/prediction/metric 변화와 어떻게 연결되는가?",
        }
    ]
)

report_sentence = (
    f"2장 모델 평가 결과, {execution_scope} 기준 test 데이터에서 source={test_source}, "
    f"rows={len(test_dataframe)}를 먼저 고정하고 threshold={baseline_metric['threshold']}를 적용해 "
    f"precision={baseline_metric['precision']}, recall={baseline_metric['recall']}, "
    f"FP={baseline_metric['FP']}, FN={baseline_metric['FN']}을 확인했습니다. "
    "따라서 Accuracy 단독 판단은 보류하고, validation 데이터 조건 차이와 threshold sweep을 함께 첨부해 "
    "데이터 조건 변화와 FP/FN trade-off를 다음 분석으로 넘깁니다."
)

print(report_sentence)


### 출력 확인: `chapter_02_model_metric_packet`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(chapter_02_model_metric_packet)


### 출력 확인: `chapter_02_handoff`

**읽는 순서**  
앞 단계 결과가 어떤 evidence 묶음으로 정리되었는지 봅니다. 다음 장으로 넘길 질문이나 제한 사항도 같이 확인합니다.

**해석 기준**  
packet과 handoff는 최종 보고서 초안에 가깝습니다. 숫자, 조건, 판단, 다음 확인이 함께 들어 있어야 합니다.

**보고서 문장 예시**  
“이 packet을 근거로 다음 장에서 serving 계약 또는 운영 관측 확인으로 넘어갑니다.”

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다. handoff에 남은 질문을 다음 장에서 계속 확인합니다.


In [ ]:
display(chapter_02_handoff)


### 7-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `package_module`, `data_source`, `execution_scope`, `feature_columns_present`를 확인합니다. CSV를 읽지 못해 embedded sample로 떨어졌다면 출력은 정상이어도 전체 데이터 기반 근거가 아니므로 보고서에는 fallback sample 기준이라고 적어야 합니다.

metric 값이 예상과 다르면 `raw data`, `label 표준화`, `feature readiness`, `score`, `threshold`, `prediction`, `label 비교`를 순서대로 확인합니다. 이 순서를 건너뛰면 label 문제, feature 결측, threshold 변경, 모델 출력 변화를 서로 혼동하게 됩니다.

validation 비교가 흔들릴 때는 모델 자체 결함으로 단정하지 않습니다. “degraded 데이터에서 feature 품질 차이가 먼저 관측되었고, 같은 score와 threshold 기준에서 FP/FN 변화가 함께 관측되었으므로 데이터 품질 신호와 threshold trade-off를 추가 확인합니다”처럼 현재 증거와 다음 action을 함께 씁니다.
